# H&M 고객 구매 패턴과 RFM 세분화

[H&M Kaggle 대회 데이터](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/data)로 구매 패턴을 분석한다. 시드 42로 고객 최대 10,000명을 추출하고, 거래 CSV를 나누어 읽으면서 선정 고객의 전체 기간 거래를 보존한다. 표본 내 구매 활동이 확인된 고객만 분석에 포함된다.

In [1]:
import sys
from pathlib import Path

from IPython.display import Markdown, display

root = Path.cwd()
if not (root / "src" / "pipeline.py").is_file():
    root = root.parent
sys.path.insert(0, str(root))

from src.pipeline import DataAnalyzer

analyzer = DataAnalyzer(root / "data" / "raw")
purchases = analyzer.load_data()
assert purchases.shape[0] >= 1000 and purchases.shape[1] >= 8
assert purchases["t_dat"].dtype.kind == "M"
assert purchases["sales_channel_id"].dtype.name == "category"
assert purchases["price"].dtype.kind in "fiu"
display(purchases.drop(columns=["customer_id", "article_id", "prod_name"]).head())
purchases.info()
display(purchases.describe())
display(purchases.isna().sum().to_frame("Missing"))

<class 'pandas.DataFrame'>
RangeIndex: 226767 entries, 0 to 226766
Data columns (total 13 columns):
 #   Column                  Non-Null Count   Dtype         
---  ------                  --------------   -----         
 0   t_dat                   226767 non-null  datetime64[us]
 1   customer_id             226767 non-null  string        
 2   article_id              226767 non-null  string        
 3   price                   226767 non-null  float64       
 4   sales_channel_id        226767 non-null  category      
 5   club_member_status      226307 non-null  category      
 6   fashion_news_frequency  225750 non-null  category      
 7   age                     225883 non-null  float64       
 8   prod_name               226767 non-null  string        
 9   product_type_name       226767 non-null  category      
 10  product_group_name      226767 non-null  category      
 11  colour_group_name       226767 non-null  category      
 12  index_group_name        226767 non-null  

,t_dat,price,age
count,226767,226767.000000,225883.000000
mean,2019-09-15 05:34:44.827157,0.027572,35.739715
min,2018-09-20 00:00:00,0.000390,16.000000
25%,2019-03-28 00:00:00,0.015237,25.000000
50%,2019-08-23 00:00:00,0.025407,31.000000
75%,2020-03-27 00:00:00,0.033881,47.000000
max,2020-09-22 00:00:00,0.493220,92.000000
std,NaN,0.018745,12.908585


,Missing
t_dat,0
customer_id,0
article_id,0
price,0
sales_channel_id,0
club_member_status,460
fashion_news_frequency,1017
age,884
prod_name,0
product_type_name,0


## 결측치와 상품 피처

고객의 나이는 거래 횟수에 따라 가중되지 않도록 고객당 한 행으로 계산한다. 멤버십 상태별 중앙값으로 대치하고, 그룹에 관측값이 없으면 전체 고객 중앙값을 사용한다. 원본 나이는 대치 여부를 확인할 수 있도록 보존한다.

상품명에서 글자 수와 단어 수를 구한다. 표본 거래에 등장하는 상품 중 로컬에서 이미지를 찾을 수 있는 상품을 시드 42로 최대 500개 선택한다. 각 이미지의 가로·세로 전체에서 32개 픽셀 위치를 균등하게 골라 RGB 배열을 만들고, 0~1로 정규화하여 평균과 모집단 표준편차를 계산한다. 회색조는 같은 값을 RGB 세 채널에 사용한다. 추출하지 않은 이미지 피처는 결측으로 유지한다.

In [ ]:
customers = purchases.drop_duplicates("customer_id").copy()
customers["age_filled"] = analyzer.handle_missing_values(
    customers, "age", "club_member_status"
)
display(customers[["age", "age_filled"]].isna().agg(["sum", "mean"]).T)
display(customers.groupby("club_member_status", observed=True, dropna=False)["age"].agg(
    customers="size", observed_ages="count", median="median"
))

purchases["age_filled"] = purchases["customer_id"].map(
    customers.set_index("customer_id")["age_filled"]
)
purchases = analyzer.add_text_features(purchases)
purchases = analyzer.add_image_features(purchases)
display(purchases[["product_name_length", "product_name_word_count", "image_mean", "image_std"]].describe())

## IQR 가격 이상치

원본 거래 가격에서 Q1과 Q3를 계산하고, IQR의 1.5배를 기준으로 탐지한다. 시각 비교용 가격만 상한과 하한으로 제한한다. 고가 구매가 실제일 수 있으므로 RFM과 월별 금액 합계는 원본 가격을 사용한다.

In [ ]:
q1, q3, lower, upper, price_outliers = analyzer.detect_outliers(purchases, "price")
purchases["price_clipped"] = purchases["price"].clip(lower, upper)
display({
    "Q1": q1, "Q3": q3, "IQR": q3 - q1, "Lower": lower, "Upper": upper,
    "Outliers": int(price_outliers.sum()), "Outlier share": price_outliers.mean(),
})